In [ ]:
import pandas as pd
import torch_directml
import mlflow
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import os

u:\CODE\ReelSense\etl\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\catas\AppData\Local\Temp\ipykernel_11420\2371018033.py:4: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


In [ ]:
os.environ["HF_HOME"]= "U:\\CODE\\models"
mlflow.set_experiment("ReelSense-Experiment")

device = torch_directml.device()

In [3]:
print(device)

privateuseone:0


In [5]:
from sentence_transformers.trainer import TrainerCallback
class EnhancedMLflowCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            # Registrar el Loss (lo que ya hacíamos)
            if "loss" in logs:
                mlflow.log_metric("train_loss", logs["loss"], step=state.global_step)
            
            # Registrar el Learning Rate (fundamental para ver el warmup)
            if "learning_rate" in logs:
                mlflow.log_metric("learning_rate", logs["learning_rate"], step=state.global_step)
            
            # Registrar la época actual (útil en entrenamientos largos)
            if "epoch" in logs:
                mlflow.log_metric("epoch", logs["epoch"], step=state.global_step)

C:\Users\catas\AppData\Local\Temp\ipykernel_11420\1019224659.py:1: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import TrainerCallback


In [ ]:
with mlflow.start_run(run_name="ReelSenseMovie_Embeddings_FineTuning"):
    mlflow.log_param("model_name", "all-MiniLM-L6-v2")
    mlflow.log_param("dataset", "Letterboxd_Movies")
    mlflow.log_param("sample_size", 20000)

    print("Loading dataset...")

    movies = pd.read_csv("../data/movies.csv/movies.csv").dropna(subset=["name", "description"]).head(20000)

    train_examples = [InputExample(texts=[r['name'], r['description']]) for _, r in movies.iterrows()]

    model  = SentenceTransformer('all-MiniLM-L6-v2', device=device)

    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)
    train_loss = losses.MultipleNegativesRankingLoss(model)
    print("Training in Radeon 7600")

    mlflowcallback = EnhancedMLflowCallback()

    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=1,
        warmup_steps=100,
        show_progress_bar=True,
        callback = mlflowcallback
    )
    model_path = "./models/reelsense-movie-embeddings"
    model.save(model_path)
    mlflow.log_artifacts(model_path, artifact_path="model_output")

    print("Model fine-tuning completed and saved to /models/reelsense-movie-embeddings")

Loading dataset...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1790.60it/s]


Training in Radeon 7600


Step,Training Loss


KeyboardInterrupt: 

In [11]:
from transformers import AutoTokenizer

enriched_movies = pd.read_csv("../data/movies_enriched.csv")

# Cargar la herramienta que cuenta los tokens
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Sacar el primer "Super Texto"
texto_prueba = enriched_movies[['name', 'description', 'main_actors', 'main_directors', 'main_genres', 'main_themes']].iloc[0]
texto_prueba = ' '.join(texto_prueba.astype(str))
tokens = tokenizer.encode(texto_prueba)

print(f"Caracteres: {len(texto_prueba)}")
print(f"Tokens reales: {len(tokens)}")

Caracteres: 614
Tokens reales: 126
